# Tiền xử lý dữ liệu VnExpress — HARNN

**Input:** `raw_data.json` — dữ liệu crawl thô  
**Output:** `dataset.json` · `vocab.json` · `label_map.json`

| Cell | Nhiệm vụ |
|------|----------|
| 1 | Cấu hình đường dẫn & tham số |
| 2 | Hàm text processing |
| 3 | Load → tokenize → encode label |
| 4 | Build vocab → lưu file |
| 5 | Kiểm tra kết quả |


## Cell 1 — Cấu hình


In [ ]:
import json, re
from collections import Counter, defaultdict
from pathlib import Path

# ── Đường dẫn ─────────────────────────────────────────────────────────────
RAW_FILE = r'C:\Users\Admin\Documents\nlp\NLP_Project\data\raw\raw_data.json' 

PROJECT_DIR = Path(r'C:\Users\Admin\Documents\nlp\NLP_Project')
PROCESS_DIR = PROJECT_DIR / 'data' / 'process_data'
OUT_DATASET = PROCESS_DIR / 'dataset.json'
OUT_VOCAB = PROCESS_DIR / 'vocab.json'
OUT_LABEL_MAP = PROCESS_DIR / 'label_map.json'
STOPWORDS_FILE = PROJECT_DIR / 'data' / 'dictionary' / 'vietnamese-stopwords.txt'

# ── Tham số ───────────────────────────────────────────────────────────────
MIN_TOKENS      = 20   # bỏ bài có ít hơn N token
VOCAB_MIN_COUNT = 3    # bỏ từ xuất hiện ít hơn N lần

# Tạo thư mục output tự động: data/process_data
PROCESS_DIR.mkdir(parents=True, exist_ok=True)

print('Config OK')
print(f'Input (absolute): {RAW_FILE}')
print(f'Output folder   : {PROCESS_DIR}')


## Cell 2 — Text processing


In [ ]:
STOPWORDS_DASH_FILE = PROJECT_DIR / 'data' / 'dictionary' / 'vietnamese-stopwords-dash.txt'

with open(STOPWORDS_FILE,      encoding='utf-8') as f:
    sw1 = {line.strip() for line in f if line.strip()}
with open(STOPWORDS_DASH_FILE, encoding='utf-8') as f:
    sw2 = {line.strip() for line in f if line.strip()}

STOPWORDS = sw1 | sw2
print(f'File gốc  : {len(sw1)} từ')
print(f'File dash : {len(sw2)} từ')
print(f'Tổng      : {len(STOPWORDS)} từ')


def clean(text: str) -> str:
    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip().lower()


def tokenize(text: str) -> list[str]:
    try:
        from underthesea import word_tokenize
        tokens = word_tokenize(clean(text), format='text').split()
    except ImportError:
        tokens = clean(text).split()
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]


def multihot(label: str, idx_map: dict) -> list[int]:
    vec = [0] * len(idx_map)
    if label and label in idx_map:
        vec[idx_map[label]] = 1
    return vec


sample = 'Cứu được người đuối nước khi livestream lúc 12h30 ngày 17/3'
print(f'\nInput : {sample}')
print(f'Tokens: {tokenize(sample)}')


## Cell 3 — Load → Build LABEL_MAP → tokenize → encode label


In [ ]:
with open(RAW_FILE, encoding='utf-8') as f:
    raw = json.load(f)
print(f'Loaded {len(raw):,} bài')

LABEL_MAP = {'l1': {}, 'l2': {}, 'l3': {}}
unique_labels = {'l1': set(), 'l2': set(), 'l3': set()}
for a in raw:
    for level, key in [('l1', 'label_l1'), ('l2', 'label_l2'), ('l3', 'label_l3')]:
        val = a.get(key, '')
        if val:
            unique_labels[level].add(val)

for level in ['l1', 'l2', 'l3']:
    for idx, val in enumerate(sorted(unique_labels[level])):
        LABEL_MAP[level][val] = idx

print(f"L1: {len(LABEL_MAP['l1'])}  L2: {len(LABEL_MAP['l2'])}  L3: {len(LABEL_MAP['l3'])}")

articles = []
skipped  = 0

for i, a in enumerate(raw):
    l1 = a.get('label_l1', '')
    l2 = a.get('label_l2', '')
    l3 = a.get('label_l3', '')

    if not l1:
        skipped += 1
        continue

    # Tokenize
    tokens = tokenize(a.get('title', '') + ' ' + a.get('content', ''))
    if len(tokens) < MIN_TOKENS:
        skipped += 1
        continue

    articles.append({
        'url':           a.get('url', ''),
        'title':         a.get('title', ''),
        'tokens':        tokens,
        'labels_l1':     [l1],
        'labels_l2':     [l2] if l2 else [],
        'labels_l3':     [l3] if l3 else [],
        'vec_l1':        multihot(l1, LABEL_MAP['l1']),
        'vec_l2':        multihot(l2, LABEL_MAP['l2']),
        'vec_l3':        multihot(l3, LABEL_MAP['l3']),
        'is_multilabel': False,
    })

    if (i + 1) % 1000 == 0:
        print(f'  {i+1}/{len(raw)}')

print(f'\nHợp lệ : {len(articles):,}')
print(f'Bỏ qua : {skipped:,}')


## Cell 4 — Build vocab → lưu file


In [ ]:
# Build vocab
counter = Counter(t for a in articles for t in a['tokens'])
vocab   = {'<PAD>': 0, '<UNK>': 1}
for word, cnt in counter.most_common():
    if cnt >= VOCAB_MIN_COUNT:
        vocab[word] = len(vocab)
print(f'Vocab: {len(vocab):,} từ (min_count={VOCAB_MIN_COUNT})')

# Lưu
with open(OUT_DATASET,   'w', encoding='utf-8') as f: json.dump(articles,  f, ensure_ascii=False, indent=2)
with open(OUT_VOCAB,     'w', encoding='utf-8') as f: json.dump(vocab,     f, ensure_ascii=False, indent=2)
with open(OUT_LABEL_MAP, 'w', encoding='utf-8') as f: json.dump(LABEL_MAP, f, ensure_ascii=False, indent=2)

print(f'\n✓ {OUT_DATASET}')
print(f'✓ {OUT_VOCAB}')
print(f'✓ {OUT_LABEL_MAP}')


## Cell 5 — Kiểm tra kết quả


In [ ]:
total   = len(articles)
has_l2  = sum(1 for a in articles if a['labels_l2'])
has_l3  = sum(1 for a in articles if a['labels_l3'])

print(f'Tổng bài   : {total:,}')
print(f'Có L2      : {has_l2:,} ({has_l2/total*100:.1f}%)')
print(f'Có L3      : {has_l3:,} ({has_l3/total*100:.1f}%)')
print(f'Vocab size : {len(vocab):,}')
print(f'vec_l1 dim : {len(articles[0]["vec_l1"])}')
print(f'vec_l2 dim : {len(articles[0]["vec_l2"])}')
print(f'vec_l3 dim : {len(articles[0]["vec_l3"])}')

# Phân phối L1
l1_dist = Counter(a['labels_l1'][0] for a in articles)
print(f'\nPhân phối L1:')
mx = max(l1_dist.values())
for lb, cnt in sorted(l1_dist.items(), key=lambda x: -x[1]):
    bar = '█' * (cnt * 30 // mx)
    print(f'  {lb:<22} {cnt:>5}  {bar}')

# 3 bài mẫu
print('\n3 bài mẫu:')
for a in articles[:3]:
    print(f"  {a['title'][:55]}")
    print(f"  L1={a['labels_l1']}  L2={a['labels_l2']}  L3={a['labels_l3']}")
    print(f"  tokens={a['tokens'][:6]}  vec_l1 sum={sum(a['vec_l1'])}")
    print()
